## Imports

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from torchvision import transforms
import os, glob
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torchvision.models as models
import torch.optim as optim
import torch.nn.functional as F
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

## Loading the dataset

In [2]:
DATA_DIR = "/kaggle/input/datasets/marccrasto/wcs-dataset"

CSV_PATH = os.path.join(DATA_DIR, "image_metadata_trainAug_testNonAug_withMeta.csv")  # adjust filename if needed

df = pd.read_csv(CSV_PATH)
print("Rows:", len(df))
print(df.columns)
df.head()

Rows: 21864
Index(['url', 'filename', 'diag', 'fst', 'partition', 'label',
       'nine_partition_label', 'three_partition_label', 'set', 'is_augmented',
       'final_image_path'],
      dtype='object')


,url,filename,diag,fst,partition,label,nine_partition_label,three_partition_label,set,is_augmented,final_image_path
0,https://www.dermaamin.com/site/images/clinical...,ur_f3_104_7e8aee92,ur,3,train,urticaria,inflammatory,non-neoplastic,train,False,all_images\ur_f3_104_7e8aee92.jpg
1,https://www.dermaamin.com/site/images/clinical...,sq-ce-ca_f2_137_2082b550,sq-ce-ca,2,train,squamous cell carcinoma,malignant epidermal,malignant,train,False,all_images\sq-ce-ca_f2_137_2082b550.jpg
2,http://atlasdermatologico.com.br/img?imageId=5007,al-co-de_f4_406_ba1017c1,al-co-de,4,train,allergic contact dermatitis,inflammatory,non-neoplastic,train,False,all_images\al-co-de_f4_406_ba1017c1.jpg
3,https://www.dermaamin.com/site/images/clinical...,demy_f3_71_cd1ae8fc,demy,3,val,dermatomyositis,inflammatory,non-neoplastic,train,False,all_images\demy_f3_71_cd1ae8fc.jpg
4,https://www.dermaamin.com/site/images/clinical...,st-jo-sy_f2_108_a59f7685,st-jo-sy,2,train,stevens johnson syndrome,inflammatory,non-neoplastic,train,False,all_images\st-jo-sy_f2_108_a59f7685.jpg


## Train, Test, Validate

In [3]:
df_train = df[df["partition"] == "train"].copy()
df_val = df[df["partition"] == "val"].copy()
df_test  = df[df["partition"] == "test"].copy()


label_to_idx = {
    "malignant": 0,
    "benign": 1,
    "non-neoplastic": 2
}
idx_to_label = {v: k for k, v in label_to_idx.items()}

df_train["y"] = df_train["three_partition_label"].map(label_to_idx)
df_val["y"]   = df_val["three_partition_label"].map(label_to_idx)
df_test["y"]  = df_test["three_partition_label"].map(label_to_idx)

df_train = df_train[["final_image_path", "y"]].copy()
df_val   = df_val[["final_image_path", "y"]].copy()
df_test  = df_test[["final_image_path", "y"]].copy()

print("Train:", len(df_train))
print("Val  :", len(df_val))
print("Test :", len(df_test))

Train: 18459
Val  : 1133
Test : 2272


## Data Augmentations

In [4]:
# Standard ImageNet normalization (because we're using ImageNet-pretrained weights)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

INPUT_SIZE = 384

train_transforms = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),   # keep consistent sizing
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_test_transforms = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

## CNN Model Creation Function

In [5]:
NUM_CLASSES = 3  # malignant, benign, non-neoplastic

def build_model(num_classes: int = NUM_CLASSES):
    # Pretrained EfficientNetV2-S
    model = models.efficientnet_v2_s(
        weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1
    )

    # Replace classifier for 3 classes
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)

    return model

model_1 = build_model()
print(model_1.classifier)

Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth


100%|██████████| 82.7M/82.7M [00:00<00:00, 220MB/s]


Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=3, bias=True)
)


## Set Device

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


## Setting the Weights

In [7]:
# counts for classes 0,1,2
counts = df_train["y"].value_counts().sort_index().values  # e.g. [malignant, benign, non-neoplastic]
weights = 1.0 / counts
weights = weights / weights.sum() * len(weights)  # normalize so avg weight ~ 1

class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

In [8]:
counts = df_train["y"].value_counts().sort_index()
print("counts:", counts.to_dict())

weights = 1.0 / counts.values
weights = weights / weights.sum() * len(weights)
print("weights:", weights)

# sanity: higher weight should go to rarer class
for i,w in enumerate(weights):
    print(i, idx_to_label[i], "weight=", float(w))

counts: {0: 6362, 1: 6381, 2: 5716}
weights: [0.96461828 0.96174604 1.07363567]
0 malignant weight= 0.964618281973336
1 benign weight= 0.9617460444937102
2 non-neoplastic weight= 1.0736356735329537


## Loading the Data with DataLoader

In [9]:
class SkinCSV_Dataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        rel_path = row["final_image_path"].replace("\\", "/")
        
        img_path = os.path.join(self.root_dir, "all_images", rel_path)

        img = Image.open(img_path).convert("RGB")
        y = int(row["y"])

        if self.transform:
            img = self.transform(img)

        return img, y

## Hyperparameter Tuning

In [10]:
import optuna
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.transforms import RandAugment, RandomErasing
from sklearn.metrics import f1_score

@torch.no_grad()
def eval_macro_f1(model, loader, device):
    model.eval()
    all_y, all_p = [], []
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        logits = model(x)
        preds = torch.argmax(logits, dim=1)
        all_y.append(y.cpu().numpy())
        all_p.append(preds.cpu().numpy())
    y_true = np.concatenate(all_y)
    y_pred = np.concatenate(all_p)
    return f1_score(y_true, y_pred, average="macro")

def make_transforms(aug_strength: str):
    # Uses your INPUT_SIZE / IMAGENET_* and avoids RandomResizedCrop
    if aug_strength == "light":
        train_transforms = transforms.Compose([
            transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(15),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    elif aug_strength == "medium":
        train_transforms = transforms.Compose([
            transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.3),
            transforms.RandomRotation(20),
            transforms.ColorJitter(0.15, 0.15, 0.15, 0.03),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    else:  # strong
        train_transforms = transforms.Compose([
            transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.35),
            transforms.RandomRotation(25),
            RandAugment(num_ops=2, magnitude=9),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
            RandomErasing(p=0.25, scale=(0.02, 0.10), ratio=(0.3, 3.3), value='random'),
        ])

    val_test_transforms = transforms.Compose([
        transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    return train_transforms, val_test_transforms

def make_balanced_sampler(df_train):
    class_counts = df_train["y"].value_counts().sort_index().to_dict()
    sample_w = df_train["y"].map(lambda c: 1.0 / class_counts[int(c)]).values
    sample_w = torch.DoubleTensor(sample_w)
    return WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)

def unfreeze_last_blocks(model, n_blocks: int):
    for p in model.parameters():
        p.requires_grad = False
    if n_blocks > 0:
        for p in model.features[-n_blocks:].parameters():
            p.requires_grad = True
    for p in model.classifier.parameters():
        p.requires_grad = True

def train_epochs_amp(model, loader_train, loader_val, device, epochs, optimizer, criterion, scheduler=None):
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
    best_f1 = -1.0
    best_state = None

    for epoch in range(epochs):
        model.train()
        for x, y in loader_train:
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                logits = model(x)
                loss = criterion(logits, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        if scheduler is not None:
            scheduler.step()

        f1 = eval_macro_f1(model, loader_val, device)
        if f1 > best_f1:
            best_f1 = f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    return best_f1, best_state

def objective(trial):
    aug_strength = trial.suggest_categorical("aug_strength", ["light", "medium", "strong"])
    unfreeze_blocks = trial.suggest_int("unfreeze_blocks", 2, 8)

    head_lr = trial.suggest_float("head_lr", 3e-4, 3e-3, log=True)
    backbone_lr = trial.suggest_float("backbone_lr", 1e-5, 5e-4, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    label_smoothing = trial.suggest_float("label_smoothing", 0.0, 0.05)
    BATCH_SIZE_trial = trial.suggest_categorical("BATCH_SIZE", [32, 48, 64])

    train_transforms, val_test_transforms = make_transforms(aug_strength)

    ds_train = SkinCSV_Dataset(df_train, DATA_DIR, transform=train_transforms)
    ds_val   = SkinCSV_Dataset(df_val,   DATA_DIR, transform=val_test_transforms)

    sampler = make_balanced_sampler(df_train)
    loader_train = DataLoader(ds_train, batch_size=BATCH_SIZE_trial, sampler=sampler, num_workers=2, pin_memory=True)
    loader_val   = DataLoader(ds_val,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

    model = build_model(num_classes=NUM_CLASSES).to(device)
    unfreeze_last_blocks(model, unfreeze_blocks)

    backbone_params, head_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if "classifier" in name:
            head_params.append(p)
        else:
            backbone_params.append(p)

    optimizer = optim.AdamW(
        [{"params": backbone_params, "lr": backbone_lr},
         {"params": head_params,     "lr": head_lr}],
        weight_decay=weight_decay
    )

    criterion = nn.CrossEntropyLoss(
        label_smoothing=label_smoothing
    )

    trial_epochs = 4  # keep short for search
    f1, _ = train_epochs_amp(model, loader_train, loader_val, device, trial_epochs, optimizer, criterion)

    trial.report(f1, step=trial_epochs)
    if trial.should_prune():
        raise optuna.TrialPruned()

    return f1

In [11]:
BEST = {'aug_strength': 'medium', 'unfreeze_blocks': 5, 'head_lr': 0.0011942275997067953, 'backbone_lr': 0.00015096624327700316, 'weight_decay': 2.559240343857156e-05, 'label_smoothing': 0.005564532618847307, 'BATCH_SIZE': 48}

## Model Building

In [12]:
train_transforms, val_test_transforms = make_transforms(BEST["aug_strength"])

ds_train = SkinCSV_Dataset(df_train, DATA_DIR, transform=train_transforms)
ds_val   = SkinCSV_Dataset(df_val,   DATA_DIR, transform=val_test_transforms)
ds_test  = SkinCSV_Dataset(df_test,  DATA_DIR, transform=val_test_transforms)

sampler = make_balanced_sampler(df_train)

loader_train = DataLoader(ds_train, batch_size=BEST["BATCH_SIZE"], sampler=sampler,
                          num_workers=2, pin_memory=True)
loader_val   = DataLoader(ds_val, batch_size=64, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(ds_test, batch_size=64, shuffle=False,
                          num_workers=2, pin_memory=True)

model = build_model(num_classes=NUM_CLASSES).to(device)
unfreeze_last_blocks(model, BEST["unfreeze_blocks"])

backbone_params, head_params = [], []
for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    if "classifier" in name:
        head_params.append(p)
    else:
        backbone_params.append(p)

optimizer = optim.AdamW(
    [{"params": backbone_params, "lr": BEST["backbone_lr"]},
     {"params": head_params,     "lr": BEST["head_lr"]}],
    weight_decay=BEST["weight_decay"]
)

criterion = nn.CrossEntropyLoss(
    label_smoothing=BEST["label_smoothing"]
)

# Train longer than Optuna trials
EPOCHS = 20

# Simple cosine schedule (good default)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

## Evaluate Functions

In [13]:
@torch.no_grad()
def eval_classification_report(model, loader, device, class_names):
    model.eval()

    y_true = []
    y_pred = []

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        logits = model(x)
        preds = torch.argmax(logits, dim=1)

        y_true.append(y.cpu().numpy())
        y_pred.append(preds.cpu().numpy())

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)

    print("\nAccuracy:", accuracy_score(y_true, y_pred))
    print("\nConfusion Matrix (rows=true, cols=pred):")
    print(confusion_matrix(y_true, y_pred))

    print("\nClassification Report:")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=class_names,
            digits=4,
            zero_division=0
        )
    )

## Mixup

In [14]:
import numpy as np
import torch
import torch.nn.functional as F

def mixup_batch(x, y, alpha=0.4):
    """Returns mixed inputs, pairs of targets, and lambda."""
    if alpha <= 0:
        return x, y, y, 1.0

    lam = np.random.beta(alpha, alpha)
    bs = x.size(0)
    idx = torch.randperm(bs, device=x.device)

    x_mix = lam * x + (1 - lam) * x[idx]
    y_a = y
    y_b = y[idx]
    return x_mix, y_a, y_b, lam

def mixup_criterion(logits, y_a, y_b, lam, class_weights=None, label_smoothing=0.0):
    # Use CE with label smoothing and optional class weights, applied to both targets
    loss_a = F.cross_entropy(logits, y_a, weight=class_weights, label_smoothing=label_smoothing)
    loss_b = F.cross_entropy(logits, y_b, weight=class_weights, label_smoothing=label_smoothing)
    return lam * loss_a + (1 - lam) * loss_b

In [15]:
def train_epochs_amp_mixup(model, loader_train, loader_val, device, epochs, optimizer,
                           class_weights, label_smoothing=0.0, mixup_alpha=0.4, scheduler=None):

    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
    best_f1 = -1.0
    best_state = None

    for epoch in range(epochs):
        model.train()
        for x, y in loader_train:
            x = x.to(device)
            y = y.to(device)

            x, y_a, y_b, lam = mixup_batch(x, y, alpha=mixup_alpha)

            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                logits = model(x)
                loss = mixup_criterion(
                    logits, y_a, y_b, lam,
                    class_weights=class_weights,
                    label_smoothing=label_smoothing
                )

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        if scheduler is not None:
            scheduler.step()

        f1 = eval_macro_f1(model, loader_val, device)
        if f1 > best_f1:
            best_f1 = f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    return best_f1, best_state

In [16]:
mixup_alpha = 0.15  # good starting point

best_f1, best_state = train_epochs_amp_mixup(
    model=model,
    loader_train=loader_train,
    loader_val=loader_val,
    device=device,
    epochs=EPOCHS,
    optimizer=optimizer,
    class_weights=None,
    label_smoothing=BEST["label_smoothing"],
    mixup_alpha=mixup_alpha,
    scheduler=scheduler
)

model.load_state_dict(best_state)

/tmp/ipykernel_55/1230852955.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
/tmp/ipykernel_55/1230852955.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
/tmp/ipykernel_55/1230852955.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
/tmp/ipykernel_55/1230852955.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
/tmp/ipykernel_55/1230852955.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is depreca

<All keys matched successfully>

In [17]:
eval_classification_report(
    model,
    test_loader,
    device,
    class_names=["malignant", "benign", "non-neoplastic"]
)


Accuracy: 0.9137323943661971

Confusion Matrix (rows=true, cols=pred):
[[ 301    2   35]
 [   0  240   60]
 [  42   57 1535]]

Classification Report:
                precision    recall  f1-score   support

     malignant     0.8776    0.8905    0.8840       338
        benign     0.8027    0.8000    0.8013       300
non-neoplastic     0.9417    0.9394    0.9406      1634

      accuracy                         0.9137      2272
     macro avg     0.8740    0.8766    0.8753      2272
  weighted avg     0.9138    0.9137    0.9138      2272

